# Conditional Diffusion for Image Colorization

In this notebook, we build a **conditional latent diffusion model** that colorizes grayscale images using a color reference image. The reference image's color palette is injected via **cross-attention** — the same mechanism Stable Diffusion uses for CLIP text embeddings.

### Learning Objectives
- Refresh the mathematical foundations of diffusion models (forward & reverse processes)
- Understand how to **condition** diffusion models on external signals (images, text)
- Learn the two main conditioning mechanisms: **channel concatenation** and **cross-attention**
- See how our approach mirrors the Stable Diffusion architecture
- Train a working colorization model on COCO images (~70 min on an L4 GPU)

### Prerequisites
You should be familiar with:
- DDPM (Denoising Diffusion Probabilistic Models)
- VAEs (Variational Autoencoders) — encoder/decoder, latent space
- Attention mechanisms and Transformers
- Latent Diffusion Models (LDMs) — the idea of running diffusion in latent space

---
## 1. Diffusion Refresher: Forward Process

A diffusion model defines a **forward (noising) process** that gradually corrupts data into pure Gaussian noise, and a **reverse (denoising) process** that learns to undo the corruption.

### Forward Process

Given a clean data sample $x_0$, we define a Markov chain that adds Gaussian noise over $T$ timesteps:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I}\right)$$

where $\beta_t \in (0, 1)$ is the **noise schedule**. We use a linear schedule: $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$, with $T = 1000$.

### Closed-Form Sampling

A key insight is that we can sample $x_t$ directly from $x_0$ without iterating through all intermediate steps. Define:

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

Then:

$$q(x_t \mid x_0) = \mathcal{N}\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) \mathbf{I}\right)$$

Using the **reparameterization trick**:

$$\boxed{x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon}, \quad \varepsilon \sim \mathcal{N}(0, \mathbf{I})$$

**Intuition**: As $t$ increases, $\bar{\alpha}_t \to 0$, so $x_t$ becomes pure noise. The signal from $x_0$ is gradually lost.

## 2. Diffusion Refresher: Reverse Process & Training

### Reverse Process

The reverse process learns to denoise, stepping from $x_t$ back to $x_{t-1}$:

$$p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\left(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 \mathbf{I}\right)$$

A neural network $\varepsilon_\theta$ parameterizes this by predicting the noise $\varepsilon$ that was added at timestep $t$.

### Simplified Training Objective ($\varepsilon$-prediction)

Instead of predicting $\mu_\theta$ directly, we train the network to predict the noise:

$$\boxed{\mathcal{L} = \mathbb{E}_{x_0,\, \varepsilon \sim \mathcal{N}(0,\mathbf{I}),\, t \sim \text{Uniform}(1,T)} \left[\| \varepsilon - \varepsilon_\theta(x_t, t) \|^2 \right]}$$

This is a simple MSE loss between the actual noise and the predicted noise.

### DDPM vs DDIM Sampling

**DDPM** (Denoising Diffusion Probabilistic Models): stochastic sampling, requires all $T = 1000$ steps. Slow but high quality.

**DDIM** (Denoising Diffusion Implicit Models): deterministic sampling with the **same trained model**, but allows far fewer steps (e.g., 50). The DDIM update rule:

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\, \varepsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}$$

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\, \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1}}\, \varepsilon_\theta(x_t, t)$$

**Key insight**: We train with the DDPM loss, but sample with DDIM for 20× faster inference.

---
## 3. Conditioning Diffusion Models

So far, diffusion models generate samples unconditionally: $p_\theta(x_{t-1} \mid x_t)$. To control generation, we condition on an external signal $c$:

$$p_\theta(x_{t-1} \mid x_t, c)$$

How do we inject $c$ into the network? There are two main mechanisms:

### Mechanism 1: Channel Concatenation

Append the condition directly to the noisy input along the channel dimension:

$$\text{input} = [x_t \;\|\; c_{\text{spatial}}] \in \mathbb{R}^{B \times (C + C') \times H \times W}$$

- **Pros**: Simple, preserves spatial alignment
- **Cons**: Condition must have the same spatial resolution as $x_t$
- **Best for**: Spatially-aligned signals (e.g., a grayscale image guiding colorization)

### Mechanism 2: Cross-Attention

Encode the condition into a sequence of tokens, then inject via cross-attention layers inside the U-Net:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

where:
- $Q = W_Q \cdot z_{\text{features}}$ — queries from U-Net intermediate features
- $K = W_K \cdot c_{\text{tokens}}$ — keys from the condition encoding
- $V = W_V \cdot c_{\text{tokens}}$ — values from the condition encoding

- **Pros**: Flexible, works with any encoded signal (text, images, audio)
- **Cons**: More parameters, condition doesn't need spatial alignment
- **Best for**: Semantic/global conditions (e.g., text prompts, style references)

### Our Design: Both!

For our colorization task, we combine both mechanisms:

| Condition | Mechanism | Rationale |
|-----------|-----------|----------|
| Grayscale image | Channel concatenation | Spatial alignment matters — the grayscale provides the structure |
| Reference color image | Cross-attention | Global color palette transfer — we want color information, not pixel alignment |

## 4. VAE & Latent Space Refresher

Running diffusion directly on $128 \times 128 \times 3$ images is expensive. **Latent Diffusion Models (LDMs)** first compress images into a low-dimensional latent space using a VAE, then run diffusion there.

### The VAE Pipeline

$$x \in \mathbb{R}^{128 \times 128 \times 3} \xrightarrow{\;\mathcal{E}\;} z \in \mathbb{R}^{16 \times 16 \times 4} \xrightarrow{\;\text{diffusion}\;} \hat{z} \in \mathbb{R}^{16 \times 16 \times 4} \xrightarrow{\;\mathcal{D}\;} \hat{x} \in \mathbb{R}^{128 \times 128 \times 3}$$

- **Encoder** $\mathcal{E}$: compresses the image by a factor of $8\times$ spatially (128 → 16) into 4 latent channels
- **Decoder** $\mathcal{D}$: reconstructs the image from latents
- **Dimensionality reduction**: $128 \times 128 \times 3 = 49{,}152$ → $16 \times 16 \times 4 = 1{,}024$ — that's **48× fewer dimensions!**

### Our VAE

We use the pre-trained Stable Diffusion VAE: [`stabilityai/sd-vae-ft-mse`](https://huggingface.co/stabilityai/sd-vae-ft-mse). It's frozen — we don't train it.

**Important detail**: The SD VAE uses a **scaling factor** of `0.18215`. After encoding, latents are multiplied by this factor to ensure they have approximately unit variance. Before decoding, we must divide by the same factor. Forgetting this produces washed-out or blown-out images.

---
## 5. Stable Diffusion Architecture & Our Approach

### Stable Diffusion

Stable Diffusion is an LDM with three components:
1. **VAE**: compresses images ↔ latents
2. **U-Net**: denoises latents, conditioned via cross-attention
3. **CLIP text encoder**: converts text prompt → sequence of 77 tokens × 768 dimensions

The CLIP tokens enter the U-Net as `encoder_hidden_states` and interact with U-Net features via cross-attention at multiple resolution levels.

### Our Simplified Version

We replace the CLIP text encoder with a **ResNet-18 image encoder** — same mechanism, different modality:

| | Stable Diffusion | Our Colorization Model |
|---|---|---|
| **Condition** | Text prompt | Reference color image |
| **Encoder** | CLIP text encoder (frozen) | ResNet-18 layer3 (frozen) + projection |
| **Token shape** | (B, 77, 768) | (B, 64, 256) |
| **Injection** | Cross-attention via `encoder_hidden_states` | Cross-attention via `encoder_hidden_states` |
| **Additional input** | — | Grayscale latent via channel concatenation |

### Full Pipeline

```
Reference Image ──► Frozen ResNet-18 (layer3) ──► Trainable Projection ──► (B, 64, 256) context tokens
                                                                                │
Grayscale ──► Frozen VAE Encoder ──► z_gray (4, 16, 16) ──────────┐            │ cross-attention
                                                                    │ concat     │ (inside U-Net)
Color Target ──► Frozen VAE Encoder ──► z_color ──► add noise ──► [x_t; z_gray] (8, 16, 16)
                                                                    │            │
                                                        UNet2DConditionModel ◄───┘
                                                                    │
                                                            ε̂ (predicted noise)
                                                                    │
                                                        DDIM sampling → VAE Decode → Colorized Image
```

The U-Net receives:
- **Input channels (8)**: 4 from the noisy color latent + 4 from the grayscale latent (concatenated)
- **Timestep $t$**: sinusoidal embedding, added inside each ResBlock
- **`encoder_hidden_states` (B, 64, 256)**: reference image tokens, injected via cross-attention layers

It outputs **4 channels**: the predicted noise $\hat{\varepsilon}$ in latent space.

---
## 6. Setup

Let's install dependencies and configure our environment. We use `diffusers` for the pre-trained VAE and the U-Net architecture, and `pycocotools` for the COCO dataset API.

In [ ]:
!pip install -q diffusers accelerate transformers pycocotools

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler

from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DDIMScheduler
from concurrent.futures import ThreadPoolExecutor, as_completed
from torchvision import transforms, models
from pycocotools.coco import COCO

from PIL import Image
import requests
import os
import io
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

---
## 7. COCO Data Loading

We use the [COCO 2017](https://cocodataset.org/) dataset, selecting **8 colorful animal categories** with rich color diversity:

> bird, cat, dog, horse, zebra, giraffe, bear, elephant

We download ~500 images per category (~4000 total).

### Pairing Strategy

For each training sample, we create a triplet:
- **Color target**: the original color image (this is what we want to reconstruct)
- **Grayscale input**: the desaturated version of the target (luminance channel replicated to 3 channels)
- **Reference image**: a *different* random image from the **same category**

By using a different image as the reference (not the target itself), the model is forced to learn **color transfer via cross-attention** — it can't simply copy pixels. It must extract the general color palette from the reference and apply it guided by the grayscale structure.

In [ ]:
# --- Download COCO annotations ---
ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOTATIONS_DIR = "/content/annotations"
IMAGES_DIR = "/content/coco_images"
ANNOTATIONS_FILE = os.path.join(ANNOTATIONS_DIR, "instances_train2017.json")

if not os.path.exists(ANNOTATIONS_FILE):
    print("Downloading COCO 2017 annotations (~252 MB)...")
    !wget -q {ANNOTATIONS_URL} -O /content/annotations.zip
    !unzip -q /content/annotations.zip -d /content/
    !rm /content/annotations.zip
    print("Done.")
else:
    print("Annotations already downloaded.")

# --- Initialize COCO API ---
coco = COCO(ANNOTATIONS_FILE)

# --- Select categories and download images ---
CATEGORIES = ["bird", "cat", "dog", "horse", "zebra", "giraffe", "bear", "elephant"]
MAX_PER_CATEGORY = 500

os.makedirs(IMAGES_DIR, exist_ok=True)
category_images = {}  # {category_name: [list of file paths]}

def download_image(meta, cat_dir):
    fpath = os.path.join(cat_dir, meta["file_name"])
    if os.path.exists(fpath):
        return fpath
    try:
        resp = requests.get(meta["coco_url"], timeout=10)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        img.save(fpath)
        return fpath
    except Exception:
        return None

for cat_name in CATEGORIES:
    cat_ids = coco.getCatIds(catNms=[cat_name])
    img_ids = coco.getImgIds(catIds=cat_ids)[:MAX_PER_CATEGORY]
    img_metas = coco.loadImgs(img_ids)

    cat_dir = os.path.join(IMAGES_DIR, cat_name)
    os.makedirs(cat_dir, exist_ok=True)

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(download_image, m, cat_dir): m for m in img_metas}
        paths = []
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"Downloading {cat_name}", leave=False):
            result = f.result()
            if result:
                paths.append(result)

    category_images[cat_name] = paths
    print(f"  {cat_name}: {len(paths)} images")

total_images = sum(len(v) for v in category_images.values())
print(f"\nTotal images downloaded: {total_images}")

In [ ]:
class COCOColorizationDataset(Dataset):
    """Dataset that returns (color, grayscale, reference) triplets from COCO images.

    - color:     original color image (the target)
    - grayscale: luminance of the target, replicated to 3 channels (VAE expects 3ch)
    - reference: a different image from the same category (color palette source)
    """

    def __init__(self, category_images: dict, image_size: int = 128):
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # -> [-1, 1]
        ])

        # Build a flat list + keep category grouping for reference pairing
        self.samples = []      # (file_path, category_index)
        self.cat_indices = []   # list of lists: indices into self.samples per category

        idx = 0
        for cat_name, paths in category_images.items():
            cat_group = []
            for p in paths:
                self.samples.append((p, len(self.cat_indices)))
                cat_group.append(idx)
                idx += 1
            self.cat_indices.append(cat_group)

    def __len__(self):
        return len(self.samples)

    def _to_grayscale_3ch(self, img_tensor):
        """Convert RGB tensor to grayscale using luminance formula, keep 3 channels."""
        # img_tensor is in [-1, 1]; convert to [0, 1] for luminance, then back
        rgb = (img_tensor + 1) / 2  # [0, 1]
        gray = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]  # (H, W)
        gray_3ch = gray.unsqueeze(0).repeat(3, 1, 1)  # (3, H, W)
        return gray_3ch * 2 - 1  # back to [-1, 1]

    def __getitem__(self, idx):
        path, cat_idx = self.samples[idx]
        img = Image.open(path).convert("RGB")
        color = self.transform(img)
        gray = self._to_grayscale_3ch(color)

        # Pick a random *different* image from the same category as the reference
        cat_group = self.cat_indices[cat_idx]
        ref_idx = idx
        while ref_idx == idx and len(cat_group) > 1:
            ref_idx = random.choice(cat_group)
        ref_path, _ = self.samples[ref_idx]
        ref_img = Image.open(ref_path).convert("RGB")
        ref = self.transform(ref_img)

        return {"color": color, "gray": gray, "ref": ref}

In [ ]:
# --- Create dataset and dataloader ---
dataset = COCOColorizationDataset(category_images, image_size=128)
dataloader = DataLoader(dataset, batch_size=14, shuffle=True, num_workers=0, pin_memory=True)
print(f"Dataset size: {len(dataset)} images")
print(f"Batches per epoch: {len(dataloader)}")

# --- Visualize sample triplets ---
def show_triplets(batch, n=5):
    """Display (grayscale, reference, color target) triplets."""
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    titles = ["Grayscale (input)", "Reference (condition)", "Color (target)"]
    for i in range(n):
        for j, key in enumerate(["gray", "ref", "color"]):
            img = batch[key][i].permute(1, 2, 0).numpy() * 0.5 + 0.5  # [-1,1] -> [0,1]
            axes[i, j].imshow(img.clip(0, 1))
            axes[i, j].set_title(titles[j] if i == 0 else "")
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()

sample_batch = next(iter(dataloader))
show_triplets(sample_batch, n=5)

---
## 8. Model Definition

Our model has four components:

| Component | Role | Source | Trainable? |
|-----------|------|--------|------------|
| **VAE** | Compress images ↔ latents (16×16×4) | `diffusers.AutoencoderKL` (frozen) | No |
| **Reference Encoder** | Encode reference image → 64 tokens × 256 dim | ResNet-18 backbone (frozen) + linear projection | Projection only (~65K params) |
| **U-Net** | Denoise latents conditioned on grayscale + reference | `diffusers.UNet2DConditionModel` | Yes (~28M params) |
| **Noise Schedulers** | Manage the noise schedule for training (DDPM) and inference (DDIM) | `diffusers.DDPMScheduler` / `DDIMScheduler` | N/A |

### How the U-Net uses conditioning

The `UNet2DConditionModel` from `diffusers` accepts an `encoder_hidden_states` argument — a tensor of shape `(B, seq_len, dim)`. Inside the U-Net, at each `CrossAttnDownBlock2D` / `CrossAttnUpBlock2D`:

1. The spatial features $z \in \mathbb{R}^{B \times C \times H \times W}$ are reshaped to a sequence: $(B, H{\cdot}W, C)$
2. **Self-attention**: $Q = K = V = z_{\text{seq}}$ — the features attend to themselves
3. **Cross-attention**: $Q = z_{\text{seq}}$, $K = V = \texttt{encoder\_hidden\_states}$ — features attend to the condition
4. **Feed-forward**: MLP applied to each token

This is identical to how CLIP text tokens condition Stable Diffusion — we just provide image tokens instead.

In [ ]:
# ============================================================
# VAE: Pre-trained Stable Diffusion VAE (frozen)
# ============================================================
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

SCALING_FACTOR = vae.config.scaling_factor  # 0.18215
print(f"VAE scaling factor: {SCALING_FACTOR}")
print(f"VAE latent channels: {vae.config.latent_channels}")  # 4


def encode_to_latent(images):
    """Encode images to VAE latent space.
    Args: images (B, 3, 128, 128) in [-1, 1]
    Returns: latents (B, 4, 16, 16)
    """
    with torch.no_grad():
        latent_dist = vae.encode(images).latent_dist
        latents = latent_dist.sample() * SCALING_FACTOR
    return latents


def decode_from_latent(latents):
    """Decode latents back to image space.
    Args: latents (B, 4, 16, 16)
    Returns: images (B, 3, 128, 128) in [-1, 1]
    """
    with torch.no_grad():
        images = vae.decode(latents / SCALING_FACTOR).sample
    return images.clamp(-1, 1)

In [ ]:
# ============================================================
# VAE Sanity Check: encode -> decode and compare
# ============================================================
test_imgs = sample_batch["color"][:4].to(device)
test_latents = encode_to_latent(test_imgs)
test_recon = decode_from_latent(test_latents)

print(f"Input shape:   {test_imgs.shape}")    # (4, 3, 128, 128)
print(f"Latent shape:  {test_latents.shape}")  # (4, 4, 16, 16)
print(f"Recon shape:   {test_recon.shape}")    # (4, 3, 128, 128)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    orig = test_imgs[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    recon = test_recon[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, i].imshow(orig.clip(0, 1))
    axes[0, i].set_title("Original" if i == 0 else "")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon.clip(0, 1))
    axes[1, i].set_title("VAE Reconstruction" if i == 0 else "")
    axes[1, i].axis("off")
plt.suptitle("VAE Encode → Decode (should be near-perfect)")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Reference Encoder (improved: unfrozen layer3, global color token, CFG null token)
# ============================================================

class ReferenceEncoder(nn.Module):
    """Encodes a reference color image into a sequence of tokens for cross-attention.

    Improvements over baseline:
      1. ResNet-18 layer3 is NOW TRAINABLE (layers 0-5 remain frozen) so the
         backbone can learn color-relevant features instead of shape-biased ones.
      2. A global average-pool branch produces 1 extra 'color summary' token,
         giving the cross-attention a holistic view of the reference palette.
      3. A learned null_token enables Classifier-Free Guidance at inference.

    Output: (B, 65, context_dim)  — 1 global + 64 spatial tokens
    """

    def __init__(self, context_dim: int = 256):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Take up to layer3 (index 7) → output: (B, 256, 8, 8) for 128×128 input
        self.backbone = nn.Sequential(*list(resnet.children())[:7])

        # Freeze conv1, bn1, relu, maxpool, layer1, layer2 (indices 0-5)
        # Leave layer3 (index 6) trainable so it learns color-relevant features
        for i, child in enumerate(self.backbone.children()):
            if i < 6:
                for p in child.parameters():
                    p.requires_grad = False
            # layer3 stays trainable

        # Spatial projection: 256 → context_dim  (64 tokens from 8×8 grid)
        self.proj = nn.Linear(256, context_dim)

        # Global color summary branch: pool all spatial info → 1 token
        self.color_gap  = nn.AdaptiveAvgPool2d(1)
        self.color_proj = nn.Linear(256, context_dim)

        # Learned null token for Classifier-Free Guidance (1 global + 64 spatial)
        self.null_token = nn.Parameter(torch.zeros(1, 65, context_dim))

    def forward(self, x):
        """Args: x (B, 3, 128, 128) — reference color image in [-1, 1]
        Returns: (B, 65, context_dim) — 1 global color token + 64 spatial tokens
        """
        x_01   = (x + 1) / 2  # [-1,1] → [0,1]
        mean   = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
        std    = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
        x_norm = (x_01 - mean) / std

        # layer3 receives gradients now (no torch.no_grad())
        feats = self.backbone(x_norm)          # (B, 256, 8, 8)
        B, C, H, W = feats.shape

        # 64 spatial tokens
        spatial = feats.reshape(B, C, H * W).permute(0, 2, 1)  # (B, 64, 256)
        spatial = self.proj(spatial)                             # (B, 64, context_dim)

        # 1 global color token
        global_ = self.color_gap(feats).reshape(B, C)           # (B, 256)
        global_ = self.color_proj(global_).unsqueeze(1)         # (B, 1, context_dim)

        tokens = torch.cat([global_, spatial], dim=1)           # (B, 65, context_dim)
        return tokens


### U-Net Configuration

We use `UNet2DConditionModel` from `diffusers` with a custom configuration:

- **`in_channels=8`**: 4 channels from the noisy color latent $x_t$ + 4 channels from the grayscale latent $z_{\text{gray}}$, concatenated along the channel dimension
- **`out_channels=4`**: predicted noise $\hat{\varepsilon}$ in latent space
- **`cross_attention_dim=256`**: matches our `ReferenceEncoder` output dimension
- **Block types**: We use `CrossAttnDownBlock2D` / `CrossAttnUpBlock2D` at the deeper levels (lower resolution, higher channels) where cross-attention is most effective and computationally feasible

#### Channel flow through the U-Net

```
Input (8, 16×16)
  ├── DownBlock2D:        128ch, 16×16
  ├── DownBlock2D:        256ch, 8×8
  ├── CrossAttnDownBlock: 256ch, 4×4  ← cross-attention with reference tokens
  ├── CrossAttnDownBlock: 512ch, 2×2  ← cross-attention with reference tokens
  │
  ├── Mid Block:          512ch, 2×2  (self-attn + cross-attn + ResBlocks)
  │
  ├── CrossAttnUpBlock:   512ch, 2×2  ← cross-attention + skip connection
  ├── CrossAttnUpBlock:   256ch, 4×4  ← cross-attention + skip connection
  ├── UpBlock2D:          256ch, 8×8  + skip connection
  └── UpBlock2D:          128ch, 16×16 + skip connection
Output (4, 16×16)
```

In [ ]:
# ============================================================
# U-Net: Conditional U-Net from diffusers (scaled up)
# ============================================================
unet = UNet2DConditionModel(
    sample_size=16,              # latent spatial size
    in_channels=8,               # 4 (noisy latent) + 4 (grayscale latent)
    out_channels=4,              # predicted noise / velocity
    layers_per_block=3,          # was 2 — deeper residual processing per level
    block_out_channels=(256, 512, 512, 768),  # was (128,256,256,512) — 3× more capacity
    attention_head_dim=8,
    down_block_types=(
        "DownBlock2D",           # 256ch, 16×16 — no attention
        "CrossAttnDownBlock2D",  # 512ch, 8×8  — cross-attention NOW ENABLED
        "CrossAttnDownBlock2D",  # 512ch, 4×4  — cross-attention with reference
        "CrossAttnDownBlock2D",  # 768ch, 2×2  — cross-attention with reference
    ),
    up_block_types=(
        "CrossAttnUpBlock2D",    # 768ch, 2×2
        "CrossAttnUpBlock2D",    # 512ch, 4×4
        "CrossAttnUpBlock2D",    # 512ch, 8×8  — cross-attention NOW ENABLED
        "UpBlock2D",             # 256ch, 16×16
    ),
    cross_attention_dim=256,      # matches ReferenceEncoder output (65 tokens × 256 dim)
).to(device)

print(f"U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}")


In [ ]:
# ============================================================
# Noise Schedulers (v-prediction + scaled-linear schedule)
# ============================================================
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="scaled_linear",  # was "linear" — better distribution of noise levels
    prediction_type="v_prediction", # was "epsilon" — weights loss more uniformly, less blur
)

ddim_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    beta_schedule="scaled_linear",
    prediction_type="v_prediction",
)

# ============================================================
# Instantiate Reference Encoder + print total trainable params
# ============================================================
ref_encoder = ReferenceEncoder(context_dim=256).to(device)

# Include ALL ref_encoder params (layer3, proj, color_proj, null_token are all trainable)
trainable_params = (
    list(unet.parameters())
    + list(ref_encoder.parameters())
)
total_trainable = sum(p.numel() for p in trainable_params if p.requires_grad)
print(f"Total trainable parameters: {total_trainable:,}")

# ============================================================
# Shape test: verify everything connects correctly
# (context is now 65 tokens, not 64)
# ============================================================
with torch.no_grad():
    dummy_x   = torch.randn(2, 8, 16, 16, device=device)
    dummy_t   = torch.randint(0, 1000, (2,), device=device)
    dummy_ctx = torch.randn(2, 65, 256, device=device)  # 65 tokens now
    dummy_out = unet(dummy_x, dummy_t, encoder_hidden_states=dummy_ctx).sample
    assert dummy_out.shape == (2, 4, 16, 16), f"Unexpected shape: {dummy_out.shape}"
    print(f"Shape test passed: input {dummy_x.shape} → output {dummy_out.shape}")


---
## 9. Training

### Training Loop Walkthrough

Each training step performs:

1. **Load pre-computed latents**: $z_{\text{color}} \in \mathbb{R}^{B \times 4 \times 16 \times 16}$ and $z_{\text{gray}} \in \mathbb{R}^{B \times 4 \times 16 \times 16}$
2. **Encode reference** via `ReferenceEncoder`: reference image $(B, 3, 128, 128)$ → context tokens $(B, 64, 256)$
3. **Sample noise**: $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$, same shape as $z_{\text{color}}$
4. **Sample timestep**: $t \sim \text{Uniform}(0, T{-}1)$
5. **Forward diffusion**: $x_t = \sqrt{\bar{\alpha}_t}\, z_{\text{color}} + \sqrt{1{-}\bar{\alpha}_t}\, \varepsilon$
6. **Concatenate**: $\text{input} = [x_t \;\|\; z_{\text{gray}}] \in \mathbb{R}^{B \times 8 \times 16 \times 16}$
7. **Predict noise**: $\hat{\varepsilon} = \text{UNet}(\text{input},\, t,\, \texttt{encoder\_hidden\_states}{=}\text{context})$
8. **Compute loss**: $\mathcal{L} = \|\varepsilon - \hat{\varepsilon}\|^2$ (MSE)
9. **Backpropagate** and update U-Net + reference projection weights

### Optimizations

- **Pre-compute latents**: We encode all images through the frozen VAE once before training, storing the latents on CPU. This avoids repeated VAE forward passes and saves ~40% training time.
- **Mixed precision (fp16)**: The U-Net forward/backward runs in float16 via `torch.cuda.amp`, reducing VRAM usage and increasing throughput by ~1.5×.
- **Gradient clipping**: We clip gradients to `max_norm=1.0` for training stability.

In [ ]:
#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

#
 
P
r
e
-
c
o
m
p
u
t
e
 
V
A
E
 
l
a
t
e
n
t
s
 
(
s
a
v
e
s
 
~
4
0
%
 
t
r
a
i
n
i
n
g
 
t
i
m
e
)

#
 
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

p
r
i
n
t
(
"
P
r
e
-
c
o
m
p
u
t
i
n
g
 
V
A
E
 
l
a
t
e
n
t
s
 
f
o
r
 
a
l
l
 
i
m
a
g
e
s
.
.
.
"
)


a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
 
=
 
[
]

a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
 
=
 
[
]

a
l
l
_
r
e
f
_
i
m
a
g
e
s
 
=
 
[
]


p
r
e
c
o
m
p
u
t
e
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(
d
a
t
a
s
e
t
,
 
b
a
t
c
h
_
s
i
z
e
=
3
2
,
 
s
h
u
f
f
l
e
=
F
a
l
s
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
2
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
)


f
o
r
 
b
a
t
c
h
 
i
n
 
t
q
d
m
(
p
r
e
c
o
m
p
u
t
e
_
l
o
a
d
e
r
,
 
d
e
s
c
=
"
E
n
c
o
d
i
n
g
"
)
:

 
 
 
 
c
o
l
o
r
_
l
a
t
e
n
t
 
=
 
e
n
c
o
d
e
_
t
o
_
l
a
t
e
n
t
(
b
a
t
c
h
[
"
c
o
l
o
r
"
]
.
t
o
(
d
e
v
i
c
e
)
)

 
 
 
 
g
r
a
y
_
l
a
t
e
n
t
 
=
 
e
n
c
o
d
e
_
t
o
_
l
a
t
e
n
t
(
b
a
t
c
h
[
"
g
r
a
y
"
]
.
t
o
(
d
e
v
i
c
e
)
)

 
 
 
 
a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
.
a
p
p
e
n
d
(
c
o
l
o
r
_
l
a
t
e
n
t
.
c
p
u
(
)
)

 
 
 
 
a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
.
a
p
p
e
n
d
(
g
r
a
y
_
l
a
t
e
n
t
.
c
p
u
(
)
)

 
 
 
 
a
l
l
_
r
e
f
_
i
m
a
g
e
s
.
a
p
p
e
n
d
(
b
a
t
c
h
[
"
r
e
f
"
]
)
 
 
#
 
k
e
e
p
 
o
n
 
C
P
U
 
a
s
 
p
i
x
e
l
 
i
m
a
g
e
s


a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
 
=
 
t
o
r
c
h
.
c
a
t
(
a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
,
 
d
i
m
=
0
)

a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
 
=
 
t
o
r
c
h
.
c
a
t
(
a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
,
 
d
i
m
=
0
)

a
l
l
_
r
e
f
_
i
m
a
g
e
s
 
=
 
t
o
r
c
h
.
c
a
t
(
a
l
l
_
r
e
f
_
i
m
a
g
e
s
,
 
d
i
m
=
0
)


p
r
i
n
t
(
f
"
C
o
l
o
r
 
l
a
t
e
n
t
s
:
 
{
a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
.
s
h
a
p
e
}
"
)
 
 
#
 
(
N
,
 
4
,
 
1
6
,
 
1
6
)

p
r
i
n
t
(
f
"
G
r
a
y
 
l
a
t
e
n
t
s
:
 
 
{
a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
.
s
h
a
p
e
}
"
)
 
 
 
#
 
(
N
,
 
4
,
 
1
6
,
 
1
6
)

p
r
i
n
t
(
f
"
R
e
f
 
i
m
a
g
e
s
:
 
 
 
 
{
a
l
l
_
r
e
f
_
i
m
a
g
e
s
.
s
h
a
p
e
}
"
)
 
 
 
 
 
#
 
(
N
,
 
3
,
 
1
2
8
,
 
1
2
8
)


#
 
C
r
e
a
t
e
 
a
 
n
e
w
 
d
a
t
a
s
e
t
 
f
r
o
m
 
p
r
e
-
c
o
m
p
u
t
e
d
 
l
a
t
e
n
t
s

c
l
a
s
s
 
L
a
t
e
n
t
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:

 
 
 
 
d
e
f
 
_
_
i
n
i
t
_
_
(
s
e
l
f
,
 
c
o
l
o
r
_
l
a
t
e
n
t
s
,
 
g
r
a
y
_
l
a
t
e
n
t
s
,
 
r
e
f
_
i
m
a
g
e
s
)
:

 
 
 
 
 
 
 
 
s
e
l
f
.
c
o
l
o
r
_
l
a
t
e
n
t
s
 
=
 
c
o
l
o
r
_
l
a
t
e
n
t
s

 
 
 
 
 
 
 
 
s
e
l
f
.
g
r
a
y
_
l
a
t
e
n
t
s
 
=
 
g
r
a
y
_
l
a
t
e
n
t
s

 
 
 
 
 
 
 
 
s
e
l
f
.
r
e
f
_
i
m
a
g
e
s
 
=
 
r
e
f
_
i
m
a
g
e
s


 
 
 
 
d
e
f
 
_
_
l
e
n
_
_
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
l
e
n
(
s
e
l
f
.
c
o
l
o
r
_
l
a
t
e
n
t
s
)


 
 
 
 
d
e
f
 
_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,
 
i
d
x
)
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
(

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
c
o
l
o
r
_
l
a
t
e
n
t
s
[
i
d
x
]
,

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
g
r
a
y
_
l
a
t
e
n
t
s
[
i
d
x
]
,

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
r
e
f
_
i
m
a
g
e
s
[
i
d
x
]
,

 
 
 
 
 
 
 
 
)


l
a
t
e
n
t
_
d
a
t
a
s
e
t
 
=
 
L
a
t
e
n
t
D
a
t
a
s
e
t
(
a
l
l
_
c
o
l
o
r
_
l
a
t
e
n
t
s
,
 
a
l
l
_
g
r
a
y
_
l
a
t
e
n
t
s
,
 
a
l
l
_
r
e
f
_
i
m
a
g
e
s
)

l
a
t
e
n
t
_
l
o
a
d
e
r
 
=
 
D
a
t
a
L
o
a
d
e
r
(
l
a
t
e
n
t
_
d
a
t
a
s
e
t
,
 
b
a
t
c
h
_
s
i
z
e
=
3
2
,
 
s
h
u
f
f
l
e
=
T
r
u
e
,
 
n
u
m
_
w
o
r
k
e
r
s
=
2
,
 
p
i
n
_
m
e
m
o
r
y
=
T
r
u
e
)

p
r
i
n
t
(
f
"
\
n
L
a
t
e
n
t
 
d
a
t
a
l
o
a
d
e
r
 
r
e
a
d
y
:
 
{
l
e
n
(
l
a
t
e
n
t
_
l
o
a
d
e
r
)
}
 
b
a
t
c
h
e
s
/
e
p
o
c
h
"
)

In [ ]:
# ============================================================
# Training Loop
# ============================================================
NUM_EPOCHS = 100
LR = 3e-4          # was 2e-4; scaled up for larger batch (32 vs 14: sqrt(32/14)≈1.51×)
GRAD_CLIP = 1.0
P_UNCOND = 0.1     # Classifier-Free Guidance dropout probability

optimizer    = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler       = GradScaler()

# Fixed test sample for periodic visualization
test_color_latent = all_color_latents[0:1].to(device)
test_gray_latent  = all_gray_latents[0:1].to(device)
test_ref_image    = all_ref_images[0:1].to(device)

losses = []
unet.train()
ref_encoder.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0

    for color_latent, gray_latent, ref_image in latent_loader:
        color_latent = color_latent.to(device)
        gray_latent  = gray_latent.to(device)
        ref_image    = ref_image.to(device)
        B = color_latent.shape[0]

        # 1. Encode reference image → context tokens for cross-attention
        context = ref_encoder(ref_image)  # (B, 65, 256)

        # 2. Classifier-Free Guidance dropout: randomly null out the reference
        #    so the model learns both conditioned and unconditional distributions.
        mask = (torch.rand(B, 1, 1, device=device) < P_UNCOND)  # (B,1,1) bool
        null = ref_encoder.null_token.expand(B, -1, -1)          # (B, 65, 256)
        context = torch.where(mask, null, context)

        # 3. Sample noise and random timesteps
        noise     = torch.randn_like(color_latent)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (B,), device=device).long()

        # 4. Forward diffusion: add noise to color latent
        noisy_latent = noise_scheduler.add_noise(color_latent, noise, timesteps)

        # 5. Concatenate noisy color latent with grayscale latent
        model_input = torch.cat([noisy_latent, gray_latent], dim=1)  # (B, 8, 16, 16)

        # 6. Predict (mixed precision)
        optimizer.zero_grad()
        with autocast():
            noise_pred = unet(model_input, timesteps,
                              encoder_hidden_states=context).sample
            # v-prediction target: more uniform loss weighting across timesteps
            target = noise_scheduler.get_velocity(color_latent, noise, timesteps)
            loss   = F.mse_loss(noise_pred, target)

        # 7. Backprop with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss  += loss.item()
        num_batches += 1

    lr_scheduler.step()
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} "
              f"| LR: {lr_scheduler.get_last_lr()[0]:.2e}")

    # Quick DDIM visualization every 20 epochs (with CFG guidance_scale=7.5)
    if (epoch + 1) % 20 == 0:
        unet.eval()
        ref_encoder.eval()
        with torch.no_grad():
            ctx          = ref_encoder(test_ref_image)                # (1, 65, 256)
            ctx_null     = ref_encoder.null_token.expand(1, -1, -1)   # (1, 65, 256)
            latent       = torch.randn(1, 4, 16, 16, device=device)
            ddim_scheduler.set_timesteps(50)
            for t in ddim_scheduler.timesteps:
                inp            = torch.cat([latent, test_gray_latent], dim=1)
                pred_cond      = unet(inp, t, encoder_hidden_states=ctx).sample
                pred_uncond    = unet(inp, t, encoder_hidden_states=ctx_null).sample
                pred           = pred_uncond + 7.5 * (pred_cond - pred_uncond)
                latent         = ddim_scheduler.step(pred, t, latent).prev_sample
            preview = decode_from_latent(latent)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        gt = decode_from_latent(test_color_latent)
        for ax, img, title in zip(axes,
            [decode_from_latent(test_gray_latent), preview, gt],
            ["Grayscale", f"Colorized (ep {epoch+1})", "Ground Truth"]):
            ax.imshow((img[0].cpu().permute(1,2,0).numpy()*0.5+0.5).clip(0,1))
            ax.set_title(title)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        unet.train()
        ref_encoder.train()

    if (epoch + 1) % 25 == 0:
        torch.save({
            "epoch": epoch,
            "unet": unet.state_dict(),
            "ref_encoder": ref_encoder.state_dict(),
            "optimizer": optimizer.state_dict(),
            "losses": losses,
        }, f"/content/checkpoint_epoch{epoch+1}.pt")
        print(f"  Checkpoint saved.")

print("\nTraining complete!")


In [ ]:
# ============================================================
# Plot training loss curve
# ============================================================
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Inference with DDIM

Now that the model is trained, we sample colorized images using **DDIM** (Denoising Diffusion Implicit Models):

1. **Start from pure noise**: $z_T \sim \mathcal{N}(0, \mathbf{I})$ in latent space $(1, 4, 16, 16)$
2. **Set timestep schedule**: 50 evenly spaced steps from $T{-}1$ down to $0$
3. **At each step**: concatenate with $z_{\text{gray}}$, predict noise, take DDIM step
4. **Decode**: pass the final denoised latent through the VAE decoder

The `DDIMScheduler.step()` method implements the DDIM update rule we saw in Section 2:
- First, it predicts $\hat{x}_0$ from the current $x_t$ and the predicted noise
- Then, it computes $x_{t-1}$ as a deterministic interpolation toward $\hat{x}_0$

Because DDIM is deterministic ($\eta = 0$), the same noise seed always produces the same output. This makes results reproducible and enables interpolation in the noise space.

In [ ]:
# ============================================================
# DDIM Sampling Function with Classifier-Free Guidance
# ============================================================

@torch.no_grad()
def sample_ddim(gray_image, ref_image, num_steps=50, guidance_scale=7.5, seed=None):
    """Generate a colorized image from a grayscale input and a color reference.

    Uses Classifier-Free Guidance (CFG): runs two UNet forward passes per step
    (conditioned + unconditional) and amplifies the difference by guidance_scale.
    Higher guidance_scale → colors adhere more strongly to the reference palette.
    Recommended range: 5–10. Try 7.5 as default.

    Args:
        gray_image:     (1, 3, 128, 128) grayscale image in [-1, 1] (3-channel)
        ref_image:      (1, 3, 128, 128) reference color image in [-1, 1]
        num_steps:      number of DDIM denoising steps
        guidance_scale: CFG scale — higher = more reference adherence (default 7.5)
        seed:           optional random seed for reproducibility
    Returns:
        colorized: (1, 3, 128, 128) in [-1, 1]
    """
    unet.eval()
    ref_encoder.eval()

    # Encode inputs
    gray_latent  = encode_to_latent(gray_image.to(device))   # (1, 4, 16, 16)
    context      = ref_encoder(ref_image.to(device))          # (1, 65, 256)
    context_null = ref_encoder.null_token.expand(1, -1, -1)   # (1, 65, 256)

    # Start from pure noise
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)
    else:
        generator = None
    latent = torch.randn(1, 4, 16, 16, device=device, generator=generator)

    # DDIM denoising loop with CFG
    ddim_scheduler.set_timesteps(num_steps)
    for t in ddim_scheduler.timesteps:
        model_input = torch.cat([latent, gray_latent], dim=1)  # (1, 8, 16, 16)

        # Conditioned and unconditional predictions
        noise_pred_cond   = unet(model_input, t, encoder_hidden_states=context).sample
        noise_pred_uncond = unet(model_input, t, encoder_hidden_states=context_null).sample

        # CFG: amplify the reference conditioning signal
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

        latent = ddim_scheduler.step(noise_pred, t, latent).prev_sample

    # Decode to pixel space
    colorized = decode_from_latent(latent)
    return colorized


In [ ]:
# ============================================================
# Run inference on test samples
# ============================================================
NUM_TEST = 6

# Sample random indices from the dataset
test_indices = random.sample(range(len(dataset)), NUM_TEST)

fig, axes = plt.subplots(NUM_TEST, 4, figsize=(14, 3.5 * NUM_TEST))
col_titles = ["Grayscale (input)", "Reference (condition)", "Colorized (output)", "Ground Truth"]

for row, idx in enumerate(test_indices):
    sample = dataset[idx]
    gray = sample["gray"].unsqueeze(0)    # (1, 3, 128, 128)
    ref = sample["ref"].unsqueeze(0)      # (1, 3, 128, 128)
    color = sample["color"].unsqueeze(0)  # (1, 3, 128, 128)

    colorized = sample_ddim(gray, ref, num_steps=50, seed=42)

    images = [gray, ref, colorized.cpu(), color]
    for col, (img, title) in enumerate(zip(images, col_titles)):
        arr = img[0].permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[row, col].imshow(arr.clip(0, 1))
        if row == 0:
            axes[row, col].set_title(title, fontsize=12)
        axes[row, col].axis("off")

plt.suptitle("Conditional Diffusion Colorization Results", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Ablation: Same grayscale, different references
# This proves the model uses cross-attention conditioning!
# ============================================================
NUM_REFS = 4

# Pick one grayscale image
base_sample = dataset[0]
gray = base_sample["gray"].unsqueeze(0)

# Pick references from different categories
ref_indices = random.sample(range(len(dataset)), NUM_REFS)

fig, axes = plt.subplots(1, NUM_REFS + 1, figsize=(4 * (NUM_REFS + 1), 4))

# Show the grayscale input
axes[0].imshow((gray[0].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
axes[0].set_title("Grayscale input", fontsize=11)
axes[0].axis("off")

for i, ref_idx in enumerate(ref_indices):
    ref = dataset[ref_idx]["color"].unsqueeze(0)  # use color as reference
    colorized = sample_ddim(gray, ref, num_steps=50, seed=42)

    # Show colorized output with the reference as a small inset
    arr = colorized[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[i + 1].imshow(arr.clip(0, 1))

    # Add reference image as inset
    ref_arr = ref[0].permute(1, 2, 0).numpy() * 0.5 + 0.5
    inset = axes[i + 1].inset_axes([0.65, 0.65, 0.33, 0.33])
    inset.imshow(ref_arr.clip(0, 1))
    inset.axis("off")
    inset.set_title("ref", fontsize=8)

    axes[i + 1].set_title(f"Reference {i+1}", fontsize=11)
    axes[i + 1].axis("off")

plt.suptitle("Same grayscale + different references → different colorizations", fontsize=13)
plt.tight_layout()
plt.show()

---
## 11. Discussion

### What We Built

We built a **conditional latent diffusion model** for image colorization that uses the same conditioning mechanism as Stable Diffusion:

- **Frozen VAE**: compresses 128×128 images into 16×16×4 latents, enabling efficient diffusion
- **Channel concatenation**: provides the grayscale structural guide, maintaining spatial alignment
- **Cross-attention conditioning**: injects the reference image's color palette via `encoder_hidden_states`, allowing the U-Net to selectively attend to relevant color regions

### Connection to Stable Diffusion

Our pipeline is a faithful simplification of the Stable Diffusion architecture. The key substitution:

| Stable Diffusion | Our Model |
|---|---|
| CLIP text encoder → (B, 77, 768) | ResNet-18 image encoder → (B, 64, 256) |
| Text prompt describes desired content | Reference image provides desired colors |
| Generates from text | Colorizes from grayscale + reference |

The mechanism is identical: condition tokens enter through `encoder_hidden_states` and interact with U-Net features via cross-attention. The model learns *what* to attend to in the condition (color information) and *where* to apply it (guided by the grayscale structure).

### Potential Failure Modes

- **Color bleeding**: colors from the reference may spread to incorrect regions
- **Reference ignored**: the model may learn to produce "average" colors, underusing the reference
- **Grayscale passthrough**: the model may output a slightly tinted grayscale instead of rich colors

These can be addressed with more training data, higher capacity, multi-level cross-attention, or classifier-free guidance.

## 12. Extension Exercises

Here are several directions to deepen your understanding:

1. **CLIP image encoder**: Replace the ResNet-18 reference encoder with [CLIP ViT](https://huggingface.co/openai/clip-vit-base-patch32). Does semantic understanding of the reference image improve colorization quality?

2. **Classifier-free guidance**: During training, randomly replace the reference context with zeros 10% of the time (`context = torch.zeros_like(context)`). At inference, blend conditional and unconditional predictions: $\hat{\varepsilon} = \varepsilon_{\text{uncond}} + s \cdot (\varepsilon_{\text{cond}} - \varepsilon_{\text{uncond}})$, where $s > 1$ amplifies the effect of conditioning.

3. **Higher resolution**: Increase from 128×128 to 256×256. The latent space becomes 32×32×4. How does this affect training time and quality?

4. **Perceptual loss**: Add an LPIPS perceptual loss alongside MSE. Does this improve the visual quality of colorized images?

5. **Quantitative evaluation**: Compute FID (Fréchet Inception Distance) or LPIPS between colorized outputs and ground truth. How does performance vary with the number of DDIM steps (10, 25, 50, 100)?